# unit06 レッスン: 名寄せ / エンティティ解決 — 古典から深層への橋

**題材** — 5つの Web サイトから集めた商品レコードについて、2行が**同じ商品**を指すか判定する。
名寄せ(entity resolution)は、検索結果の統合、顧客マスタの重複除去、商品価格比較で毎日のように現れる仕事だ。

unit05 ではテキストを TF-IDF ベクトルにした。今日はそこへ、既習の2次元 DP、候補削減、
コサイン類似度、意味埋め込みをつなぎ、最後に「どの道具をどの表記ゆれへ使うか」を判断できるようにする。

## このレッスンを終えると作れるようになるもの

1. 名寄せを「レコードのペアに対する二値分類」として定式化し、全ペアを作らず **blocking** で候補を絞れる
2. レーベンシュタイン編集距離と文字 n-gram Jaccard を、何を測る量か説明して使える
3. TF-IDF の L2 正規化済みベクトルでは、内積がコサイン類似度になる理由を説明できる
4. `(batch, seq_len, dim)` の埋め込みを attention mask 付き mean pooling で `(batch, dim)` に集約できる
5. 文字列手法が崩れる「同義語での言い換え」を、埋め込みが拾う理由を数字で説明できる

所要の目安: **20〜25分**。このあと演習 `ex01`〜`ex04` が続く。

## 読み方

各概念を **見る → 予測する → 変える → 書く → チェック** の順で進む。
`# ここに書く` を空のまま実行しても止まらず、チェックポイントが `[NG]` と修正の方向を返す。

In [ ]:
# STEP 1: 読み方の処理を実行し、出力を照合する
# ===== セットアップ: このセルを最初に1回だけ実行する =====
from pathlib import Path
import unicodedata

import numpy as np
import pandas as pd

pd.set_option("display.width", 150)
pd.set_option("display.max_columns", 20)

# notebook をユニット直下から開いても、リポジトリのルートから開いても動く。
DATA = Path("data")
if not (DATA / "records.csv").exists():
    DATA = Path("courses/kaggle-sprint/unit06-entity-resolution/data")
assert (DATA / "records.csv").exists(), f"records.csv が見つかりません: {DATA.resolve()}"
assert (DATA / "vocab.csv").exists(), f"vocab.csv が見つかりません: {DATA.resolve()}"

print("pandas:", pd.__version__, "/ numpy:", np.__version__)
print("DATA =", DATA.resolve())


# ---------- 採点ヘルパー(中身は読まなくてよい) ----------
def check(name, actual, expected, hint=""):
    import numpy as _np
    try:
        ok = actual is not None and bool(_np.all(_np.isclose(_np.asarray(actual, dtype=float), _np.asarray(expected, dtype=float))))
    except (TypeError, ValueError):
        ok = actual == expected
    if ok:
        print(f"[OK] {name}: 正解!")
    else:
        print(f"[NG] {name}: 期待値 {expected!r} / 実際 {actual!r}")
        if hint:
            print(f"     ヒント: {hint}")
    return ok


def call_safely(fn, *args, **kwargs):
    # 未完成の関数を呼んでも notebook を止めない。
    if not callable(fn):
        return None
    try:
        return fn(*args, **kwargs)
    except Exception as e:
        print(f"     (関数内で {type(e).__name__}: {e})")
        return None


def shape_safely(x):
    try:
        return tuple(x.shape)
    except Exception:
        return None


print("セットアップ完了。ヘルパー: check / call_safely / shape_safely")

---
# 概念1 — ペア化と blocking

## ① なぜ: 1340行でも、総当たりは約90万ペアになる

名寄せは「各行にクラス名を付ける」問題ではない。**2つの行を受け取り、同じ実体なら1、違えば0**を返す
二値分類として考える。ところが $n$ 行の全ペアは $n(n-1)/2$ 個。日次数万件の Web 収集データでは、
類似度を計算する前に組合せ数だけで詰む。

そこで、ブランド・タイトル先頭・価格帯などの粗いキーで**同じバケツに入った行だけ**を候補にする。
これが blocking。比較サイトの候補生成や、顧客マスタ統合の最初の関門で使われる。

## ② 解説: `Dictionary<string, List<int>>` で候補を作る

C# なら `Dictionary<string, List<int>>` にブロックキーごとの行番号を集め、各 `List<int>` の中だけ二重ループする。
Python の `dict.setdefault(key, [])` は「キーがなければ空リストを作り、そのリストを返す」メソッドで、
C# の `TryGetValue` と初期化を1行にしたものだ。

| 用語 / API | 何をするものか | 戻り値・注意 |
|---|---|---|
| candidate pair | 詳しい類似度を計算する対象の `(i, j)` | 同じ組を重複させないため `set` に入れる |
| blocking key | 行を粗く同じバケツへ寄せるキー | きつ過ぎると正例を落とし、緩過ぎると計算量が戻る |
| `dict.setdefault(k, [])` | `k` の値を取得し、無ければ `[]` を登録 | 登録されたリストを返す |
| 複数キーの和集合 | ブランド候補 **または** 価格帯候補を残す | recall を上げやすいが候補数も増える |

このデータの正本: **897,130 全ペアのうち正例は1,136件(0.127%)**。
blocking 後は **130,584ペア(14.6%)、正例 recall 88.7%**。
計算を約1/7にする代わりに、正例を約11%取りこぼしている。

In [ ]:
# GOAL: バケツ内だけで候補ペアを作る流れを、小さな表で見る

records = pd.read_csv(DATA / "records.csv")
print("records:", records.shape, "/ 商品数:", records["product_key"].nunique())
print("先に shape を確認:", records[["record_id", "source", "title", "brand", "price"]].shape)


# STEP 1: 同じキーの行番号を集め、バケツ内の組だけを作る。
def pairs_from_key(values):
    buckets = {}
    for i, key in enumerate(values):
        buckets.setdefault(key, []).append(i)
    pairs = set()
    for ids in buckets.values():
        for p, i in enumerate(ids):
            for j in ids[p + 1:]:
                pairs.add((i, j))
    return pairs


# STEP 2: 正解が見える極小表で、ブランドだけの blocking を観察する。
mini = pd.DataFrame({
    "title": ["KURO A100 イヤホン", "KURO A100 無線イヤフォン", "KURO B900 冷蔵庫", "ソライロ B900 冷蔵庫"],
    "brand": ["KURO", "KURO", "KURO", "ソライロ"],
    "price_band": [10, 10, 80, 80],
    "true_entity": ["P1", "P1", "P2", "P2"],
})
brand_pairs = pairs_from_key(mini["brand"])
print("\nmini shape:", mini.shape)
print(mini)
print("\nブランドだけの候補:", sorted(brand_pairs))
print("全6ペアを比べずに済む一方、ブランド表記が違う P2 の正例 (2, 3) は落ちる。")

## ④ 予測: blocking key を増やすと何が起きる?

次のセルでは、ブランド候補に**価格帯が同じ候補**を和集合で足す。

1. 候補数は3より増える? 減る?
2. ブランドが違う正例 `(2, 3)` は候補へ戻る?
3. 代わりに、新しい負例も候補へ混ざる可能性はある?

実行前に「recall と計算量のどちらがどう動くか」を言葉で予測しよう。

In [ ]:
# GOAL: 複数 blocking key の和集合で recall と候補数が同時に増えることを見る

price_pairs = pairs_from_key(mini["price_band"])
union_pairs = brand_pairs | price_pairs     # set の | は和集合。C# の Union(...).ToHashSet() 相当

print("ブランド候補:", sorted(brand_pairs), "件数=", len(brand_pairs))
print("価格帯候補  :", sorted(price_pairs), "件数=", len(price_pairs))
print("和集合      :", sorted(union_pairs), "件数=", len(union_pairs))
print("(2, 3) が復活:", (2, 3) in union_pairs)
print("\n本番は 897,130 → 130,584 ペア。候補率14.6%、正例 recall 88.7%。")
print("blocking は正解判定ではない。『詳しく調べる価値がある組』を安く残す工程。")

## ⑥ 書いてみる: blocking の費用と取りこぼしを数値にする

次の3変数を埋めよう。

- `total_pairs`: 1340行の全ペア数。式は `n * (n - 1) // 2`
- `candidate_fraction`: blocking 後の候補率 `130584 / total_pairs`
- `missed_fraction`: 正例 recall 0.887 から計算した取りこぼし率

候補率は小数第4位、取りこぼし率は小数第3位へ `round` する。3行で書ける。

In [ ]:
# STEP 4: ⑥ 書いてみる: blocking の費用と取りこぼしを数値にするの処理を実行し、出力を照合する
total_pairs = None
candidate_fraction = None
missed_fraction = None
# ここに書く(ヒント: ペア数は組合せ。recall の反対側が取りこぼし率)

print("全ペア:", total_pairs)
print("候補率:", candidate_fraction)
print("取りこぼし率:", missed_fraction)

In [ ]:
# STEP 5: ⑥ 書いてみる: blocking の費用と取りこぼしを数値にするの処理を実行し、出力を照合する
# ===== チェックポイント A: ペア数と blocking =====
check("A-1 全ペア数", total_pairs, 897130,
      hint="n=1340 として n*(n-1)//2。順序違い (i,j)/(j,i) は同じ1ペア。")
check("A-2 blocking 後の候補率", candidate_fraction, 0.1456,
      hint="130584 / total_pairs を round(..., 4)。約14.6%になる。")
check("A-3 正例の取りこぼし率", missed_fraction, 0.113,
      hint="1 - 0.887 を round(..., 3)。")

---
# 概念2 — 編集距離と Jaccard

## ① なぜ: 表層の揺れは、まず安い文字列手法で拾う

型番のハイフン有無、全角/半角、1文字のタイプミス、語順の入れ替えは、意味モデルを呼ばなくても捕まる。
blocking 後の十万ペアすべてへ重いモデルを当てる前に、**安く説明可能な類似度**を置くと、
ベースラインにもデバッグにもなる。

ここで AtCoder で身につけた2次元 DP がそのまま実務の武器になる。

## ② 解説: 「並び」と「集合」を別々に測る

**レーベンシュタイン編集距離**は、一方の文字列を他方へ変える最小の挿入・削除・置換回数。
`dp[i, j]` を「先頭 `i` 文字と先頭 `j` 文字の距離」とする、既習の2次元 DP そのものだ。
距離を `1 - distance / max(len(a), len(b))` にすると、1に近いほど似ているスコアになる。

**Jaccard 係数**は集合 $A,B$ について $|A \cap B| / |A \cup B|$。
文字2-gram 集合を使えば、隣り合う2文字の重なりを測れる。語順を完全には無視しないが、
編集距離ほど全体の位置ずれに厳しくない。

| 道具 | 主に測るもの | 苦手 |
|---|---|---|
| 編集距離 DP | 文字列全体を直す手数 | 語順の大変更、同義語 |
| 文字2-gram Jaccard | 局所的な文字片の重なり | 共通文字のない言い換え |
| `unicodedata.normalize("NFKC", s)` | 全角/半角など見た目だけの差を統一 | 意味の違いは扱わない |

In [ ]:
# GOAL: 編集距離 DP の表を作り、表記正規化の前後でスコアがどう動くかを見る

def levenshtein(a, b, return_table=False):
    dp = np.zeros((len(a) + 1, len(b) + 1), dtype=int)
    dp[:, 0] = np.arange(len(a) + 1)
    dp[0, :] = np.arange(len(b) + 1)
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            cost = 0 if a[i - 1] == b[j - 1] else 1
            dp[i, j] = min(dp[i - 1, j] + 1,       # 削除
                           dp[i, j - 1] + 1,       # 挿入
                           dp[i - 1, j - 1] + cost) # 一致または置換
    return dp if return_table else int(dp[-1, -1])


def compact(text):
    s = unicodedata.normalize("NFKC", str(text)).lower()
    return s.replace("-", "").replace(" ", "")


left = "ＫＵＲＯ A123-45 イヤホン"
right = "KURO A12345 イヤホン"
print("正規化前の距離:", levenshtein(left, right))
print("正規化後:", compact(left), "/", compact(right))
print("正規化後の距離:", levenshtein(compact(left), compact(right)))

table = levenshtein("book", "back", return_table=True)
print("\nDP table shape:", table.shape, "= (len('book')+1, len('back')+1)")
print(table)

## ④ 予測: 同義語へ言い換えたらどうなる?

次のセルでは、文字2-gram Jaccard を2種類のペアへ使う。

- 層1: `ワイヤレスイヤホン` と、1文字落ちた `ワイヤレスイヤホ`
- 層3: `ワイヤレスイヤホン` と、同義語の `Bluetoothヘッドホン`

意味はどちらも近い。Jaccard は同じように高くなるだろうか。
また、2-gram を1-gramへ変えたらタイプミスへの強さはどう変わるか予測しよう。

In [ ]:
# GOAL: 文字の重なりだけでは同義語を拾えないことを実測する

def char_ngrams(s, n=2):
    s = compact(s)
    return {s[i:i+n] for i in range(max(0, len(s) - n + 1))}


def jaccard(a, b):
    union = a | b
    return 1.0 if not union else len(a & b) / len(union)


base = "ワイヤレスイヤホン"
typo = "ワイヤレスイヤホ"
synonym = "Bluetoothヘッドホン"

for n in (1, 2):
    s_typo = jaccard(char_ngrams(base, n), char_ngrams(typo, n))
    s_syn = jaccard(char_ngrams(base, n), char_ngrams(synonym, n))
    print(f"{n}-gram  タイプミス={s_typo:.3f} / 同義語={s_syn:.3f}")

print("\n本番の正例中央値: 層1 Jaccard=0.667 / 層3=0.306")
print("同じ意味でも共通文字が無ければ、文字集合には証拠が載らない。")

## ⑥ 書いてみる: 正規化編集類似度を関数にする

`normalized_edit_similarity(a, b)` を完成させよう。

1. `levenshtein(a, b)` で距離を取る
2. 分母は2文字列の長さの大きい方
3. 両方が空文字なら、同じ文字列なので `1.0`
4. それ以外は `1.0 - distance / denominator` を返す

4〜5行で書ける。ここでは `compact` を自動適用しない。前処理と類似度を分けると、どちらの効果か測れるためだ。

In [ ]:
# STEP 8: ⑥ 書いてみる: 正規化編集類似度を関数にするの処理を実行し、出力を照合する
def normalized_edit_similarity(a, b):
    # ここに書く(ヒント: 距離を長い方の文字数で割り、1から引く。空文字同士だけ先に扱う)
    return None


print("abc / adc:", call_safely(normalized_edit_similarity, "abc", "adc"))

In [ ]:
# STEP 9: ⑥ 書いてみる: 正規化編集類似度を関数にするの処理を実行し、出力を照合する
# ===== チェックポイント B: 正規化編集類似度 =====
check("B-1 完全一致", call_safely(normalized_edit_similarity, "same", "same"), 1.0,
      hint="距離0なので、1 - 0/4 = 1。")
check("B-2 1文字置換", call_safely(normalized_edit_similarity, "abc", "adc"), 0.6666666666666667,
      hint="距離は1、分母は3。")
check("B-3 片方が空", call_safely(normalized_edit_similarity, "abc", ""), 0.0,
      hint="3文字すべてを削除するので距離3、分母3。")
check("B-4 空文字どうし", call_safely(normalized_edit_similarity, "", ""), 1.0,
      hint="分母0で割る前に、同じ空文字どうしを1.0として扱う。")

---
# 概念3 — TF-IDF コサインと SVD(LSA)

## ① なぜ: 文字の位置より「重要な語の重なり」を見たい

語順が大きく入れ替わっても、ブランド名や型番が共有されていれば同一商品の有力な証拠になる。
unit05 の TF-IDF は、文書を「語ごとの重要度ベクトル」へ変換した。名寄せでは2行のベクトルの角度、
つまり**コサイン類似度**をペア特徴として使う。

実務では数万語の疎ベクトルをそのまま使うほか、SVD で低次元の密ベクトルへ圧縮し、
語の共起パターンをまとめた LSA(Latent Semantic Analysis) を候補検索に使う。

## ② 解説: L2 正規化済みなら内積がコサインになる

$$\cos(a,b)=\frac{a\cdot b}{\|a\|\|b\|}$$

`TfidfVectorizer` は「文書集合から語彙表と IDF を学び、各文書を TF-IDF の疎ベクトルへ変換する」クラス。
既定で各行を L2 ノルム1へ正規化するので、**2行の内積だけでコサイン**が出る。

`TruncatedSVD` は「疎行列を中心化せず、重要な方向だけ残して低次元の密行列へ変換する」クラス。
PCA と違って平均を引かないため、巨大な疎行列を密行列化せず扱える。

| API / 属性 | 何をするものか | shape / 注意 |
|---|---|---|
| `fit_transform(docs)` | train で語彙・IDFを決め、そのまま変換 | test へは使わない |
| `transform(docs)` | 決定済みの語彙・IDFで変換 | 列の意味が train と揃う |
| `X @ X.T` | L2 正規化済み行どうしの内積 | 全組合せを作ると $O(n^2)$。blocking 後だけに使う |
| `TruncatedSVD(n_components=k)` | 語彙軸を `k` 個の潜在方向へ圧縮 | `(n, vocab) → (n, k)` |
| `explained_variance_ratio_` | 各成分が保持した分散の割合 | 合計を見て、情報を捨て過ぎていないか確認 |

C# で言えば、疎な `Dictionary<int,float>` 特徴を、固定長 `float[k]` へ写す状態付き変換器だ。

In [ ]:
# GOAL: 小さな TF-IDF 行列を作り、shape と「内積 = コサイン」を目で確認する

def tiny_tfidf(docs):
    tokenized = [d.split() for d in docs]
    vocab = sorted({t for row in tokenized for t in row})
    counts = np.array([[row.count(t) for t in vocab] for row in tokenized], dtype=float)
    df = (counts > 0).sum(axis=0)
    idf = np.log((1 + len(docs)) / (1 + df)) + 1
    X = counts * idf
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    X = X / np.where(norms == 0, 1.0, norms)
    return X, vocab, idf


docs = [
    "kuro a123 イヤホン",
    "イヤホン kuro a123",
    "sorairo b900 冷蔵庫",
    "bluetooth ヘッドホン",
]
X_tfidf, demo_vocab, demo_idf = tiny_tfidf(docs)
print("X_tfidf shape:", X_tfidf.shape, "= (レコード数, 語彙数)")
print("各行のL2ノルム:", np.round(np.linalg.norm(X_tfidf, axis=1), 6))
print("\nコサイン行列 shape:", (X_tfidf @ X_tfidf.T).shape)
print(np.round(X_tfidf @ X_tfidf.T, 3))
print("\n0番と1番は語順が違っても同じ語を共有するので1.0。")
print("0番と3番は意味が近くても共有語がないので0.0。")

## ④ 予測: 語彙軸を2次元、1次元へ潰すとどうなる?

次のセルでは、同じ `(4, 語彙数)` 行列を NumPy の SVD で `(4, 2)` と `(4, 1)` へ圧縮する。
本番の疎行列では `TruncatedSVD` を使うが、極小データでは中身を見やすくするため密行列で確認する。

1. `k=2` と `k=1` のどちらが保持分散率は高い?
2. 出力の列数はそれぞれいくつ?
3. 次元を減らすほど検索は軽くなるが、別商品の区別はどうなりそう?

In [ ]:
# GOAL: SVD が shape と保持情報量のトレードオフであることを見る

U, singular_values, Vt = np.linalg.svd(X_tfidf, full_matrices=False)
total_energy = float(np.sum(singular_values ** 2))

for k in (2, 1):
    X_lsa = U[:, :k] * singular_values[:k]   # 各行を k 次元の密ベクトルへ
    retained = float(np.sum(singular_values[:k] ** 2) / total_energy)
    print(f"k={k}: shape={X_lsa.shape}, 保持エネルギー比={retained:.3f}")

print("\n実データなら:")
print("  svd = TruncatedSVD(n_components=128, random_state=0)")
print("  X_lsa = svd.fit_transform(X_train)   # shape を必ず print")
print("  X_test_lsa = svd.transform(X_test)   # test では fit しない")
print("圧縮は速さとメモリを買う代わりに情報を捨てる。保持率とCVで次元を決める。")

## ⑥ 書いてみる: 密ベクトルのコサイン類似度

`cosine_dense(a, b)` を完成させよう。TF-IDF、SVD後の LSA、次の埋め込みのすべてに再利用できる関数だ。

- `np.asarray(..., dtype=float)` で NumPy 配列にする
- 分子は `np.dot(a, b)`、分母は2つの `np.linalg.norm` の積
- どちらかがゼロベクトルなら `0.0`
- 戻り値は Python の `float`

5〜6行で書ける。ベクトル全体を `if a:` と判定すると真偽値が1つに決まらずエラーになるので、ノルムを見る。

In [ ]:
# STEP 12: ⑥ 書いてみる: 密ベクトルのコサイン類似度の処理を実行し、出力を照合する
def cosine_dense(a, b):
    # ここに書く(ヒント: 内積 / (aの長さ * bの長さ)。ゼロベクトルだけ先に扱う)
    return None


print("直交:", call_safely(cosine_dense, [1, 0], [0, 1]))

In [ ]:
# STEP 13: ⑥ 書いてみる: 密ベクトルのコサイン類似度の処理を実行し、出力を照合する
# ===== チェックポイント C: コサイン類似度 =====
check("C-1 同じ向き", call_safely(cosine_dense, [2, 0], [5, 0]), 1.0,
      hint="大きさが違っても向きが同じなら1。")
check("C-2 直交", call_safely(cosine_dense, [1, 0], [0, 3]), 0.0,
      hint="内積が0。")
check("C-3 45度", call_safely(cosine_dense, [1, 0], [1, 1]), 0.7071067811865475,
      hint="分子1、分母は sqrt(2)。")
check("C-4 ゼロベクトル", call_safely(cosine_dense, [0, 0], [1, 1]), 0.0,
      hint="0除算になる前に0.0を返す。")

---
# 概念4 — 意味埋め込みと mask 付き mean pooling

## ① なぜ: 共通文字がなくても、同じ意味の表現がある

`ワイヤレスイヤホン` と `Bluetoothヘッドホン`、`冷蔵庫` と `レフリジレーター`。
人には同義と分かるが、文字 n-gram も TF-IDF も共通の列をほとんど持てない。
ここで、意味の近い語が近い数値ベクトルになる**埋め込み(embedding)**が必要になる。

実務の文エンコーダは重いが、学ぶべきロジックは同じ。この教材ではダウンロードを一切せず、
`vocab.csv` の意味グループから作る seed 固定の疑似埋め込みを使う。

## ② 解説: lookup した語ベクトルを、文1本へ集約する

単語埋め込みは、C# の `Dictionary<string, float[]>` に見えるが、実体は
`(vocab_size, dim)` の行列と「token → 行番号」の辞書。token ID で行を引く操作が **embedding lookup** だ。

文には複数語があるので、`(seq_len, dim)` を `(dim,)` へまとめる **pooling** が要る。
バッチでは入力 shape が `(batch, seq_len, dim)`。長さをそろえるために入れた padding は平均から除外する。

| 量 | shape | 役割 |
|---|---|---|
| `token_embeddings` | `(batch, seq_len, dim)` | 各 token の意味ベクトル |
| `attention_mask` | `(batch, seq_len)` | 本物token=1、padding=0 |
| `attention_mask[:, :, None]` | `(batch, seq_len, 1)` | dim 軸を足し、埋め込みへブロードキャスト可能にする |
| 分子 | `(batch, dim)` | mask 後に `axis=1` で合計 |
| 分母 | `(batch, 1)` | `mask.sum(axis=1, keepdims=True)`。各文の実 token 数 |

既習の NumPy ブロードキャストがそのまま効く山場だ。**まず shape を print** してから式を書く。
`keepdims=True` は、C# なら「各バッチの件数をスカラーに潰さず、1列の行列として残す」感覚に近い。

In [ ]:
# GOAL: (batch, seq_len, dim) を mask 付き平均で (batch, dim) へ変える

token_embeddings = np.array([
    [[1, 0, 0], [1, 2, 0], [99, 99, 99], [99, 99, 99]],
    [[0, 2, 0], [0, 2, 2], [0, 2, 4], [99, 99, 99]],
], dtype=float)
attention_mask = np.array([
    [1, 1, 0, 0],
    [1, 1, 1, 0],
], dtype=float)

# STEP 1: shape を先に確認する。ここを飛ばすと axis の事故が起きる。
print("token_embeddings:", token_embeddings.shape)
print("attention_mask  :", attention_mask.shape)

# STEP 2: None で長さ1の dim 軸を足す。値は変えず shape だけ変える。
expanded_mask = attention_mask[:, :, None]
print("expanded_mask   :", expanded_mask.shape)

# STEP 3: padding を0にし、seq_len 軸を合計して実token数で割る。
masked = token_embeddings * expanded_mask
numerator = masked.sum(axis=1)
denominator = attention_mask.sum(axis=1, keepdims=True)
pooled_demo = numerator / denominator
print("numerator       :", numerator.shape)
print("denominator     :", denominator.shape)
print("pooled          :", pooled_demo.shape)
print(pooled_demo)

## ④ 予測: mask を無視した平均はどう壊れる?

③ の padding 位置には、わざと `[99, 99, 99]` を入れた。

1. `token_embeddings.mean(axis=1)` と単純平均すると、1文目と2文目のどちらがより大きく汚染される?
2. padding がゼロベクトルなら単純平均でも安全? ベクトルの**向き**と**大きさ**を分けて考えよう。
3. 余計な色名が本物tokenとして入った場合、mask では消せる? それは padding だろうか?

実行前に予測してから次へ進もう。

In [ ]:
# GOAL: mask は padding を消せるが、余計な本物tokenまでは消せないことを見る

naive = token_embeddings.mean(axis=1)
print("単純平均(padding込み):")
print(np.round(naive, 3))
print("\nmask付き平均:")
print(np.round(pooled_demo, 3))

# 意味中心 [1,0] に、関係の薄い色ベクトル [0,1] が混ざる例。
product_word = np.array([1.0, 0.0])
synonym_word = np.array([0.98, 0.02])
unrelated_color = np.array([0.0, 1.0])

synonym_only = synonym_word
with_color = np.mean([synonym_word, unrelated_color], axis=0)

def _cos(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

print("\n同義語だけとの cosine:", round(_cos(product_word, synonym_only), 3))
print("余計な色も mean pooling:", round(_cos(product_word, with_color), 3))
print("mask は padding を除く道具。意味はあるが無関係な語は残り、平均を薄める。")
print("本番でも層2の埋め込み中央値0.754が層3の0.955より低い理由がこれ。")

## ⑥ 書いてみる: batch 対応の mask 付き mean pooling

`masked_mean_pooling(token_embeddings, attention_mask)` を完成させよう。

- 入力 shape は `(batch, seq_len, dim)` と `(batch, seq_len)`
- mask に `[:, :, None]` で軸を足して掛ける
- `axis=1` が seq_len 軸
- 分母は `attention_mask.sum(axis=1, keepdims=True)`
- 返り値 shape は `(batch, dim)`

3〜4行で書ける。③ の shape 出力を見ながらでよい。

In [ ]:
# STEP 16: ⑥ 書いてみる: batch 対応の mask 付き mean poolingの処理を実行し、出力を照合する
def masked_mean_pooling(token_embeddings, attention_mask):
    # ここに書く(ヒント: mask に長さ1の dim 軸を足し、seq_len 軸を合計して実token数で割る)
    return None


pooled_answer = call_safely(masked_mean_pooling, token_embeddings, attention_mask)
print("pooled_answer shape:", shape_safely(pooled_answer))
print(pooled_answer)

In [ ]:
# STEP 17: ⑥ 書いてみる: batch 対応の mask 付き mean poolingの処理を実行し、出力を照合する
# ===== チェックポイント D: mask 付き mean pooling =====
check("D-1 出力 shape", shape_safely(pooled_answer), (2, 3),
      hint="seq_len 軸だけを消すので (batch, dim) = (2, 3)。")
check("D-2 pooling の値", pooled_answer, [[1.0, 1.0, 0.0], [0.0, 2.0, 2.0]],
      hint="padding の99は mask で0にする。1文目は2token、2文目は3tokenで割る。")
_probe_embeddings = token_embeddings.copy()
_probe_result = call_safely(masked_mean_pooling, _probe_embeddings, attention_mask)
_probe_padding = _probe_embeddings[0, 2].tolist() if _probe_result is not None else None
check("D-3 入力を破壊していない", _probe_padding, [99.0, 99.0, 99.0],
      hint="掛け算の結果を新しい配列へ受け、入力 token_embeddings は変更しない。")

---
# 統合 — 4種類の証拠を1本の名寄せパイプラインへ

データ生成スクリプトで実測済みの正例中央値を、表記ゆれの層ごとに並べる。
数字は大きいほど「同じ商品らしい」。次のセルは再計測ではなく、正本の固定値を表示する。

パイプライン全体は次の順になる。

1. blocking で候補ペアを作る
2. 各候補へ `[編集類似度, Jaccard, TF-IDF cosine, 埋め込み cosine, 価格差, 型番一致]` を作る
3. 二値分類器で同一確率を出し、precision / recall のどちらを重視するかで閾値を決める
4. 閾値を超えたペアを辺として、Union-Find / BFS で連結成分を名寄せグループにする

最後の段は既習の競プロそのもの。ただし **A≒B, B≒C でも A≠C** があり得る。
連結成分は推移的に全部を結ぶので、誤った1辺が大きな誤結合を起こすことに注意する。

In [ ]:
# GOAL: 表記ゆれの層ごとに、どの特徴が証拠を保つかを比較する

layer_scores = pd.DataFrame({
    "正規化編集類似度": [0.348, 0.300, 0.238],
    "Jaccard": [0.667, 0.455, 0.306],
    "TF-IDF cosine": [0.597, 0.495, 0.265],
    "埋め込み cosine": [1.000, 0.754, 0.955],
}, index=pd.Index([1, 2, 3], name="表記ゆれの層"))

print(layer_scores)
print("\n層1 → 層3 の変化:")
print("  Jaccard       : 0.667 → 0.306")
print("  TF-IDF cosine : 0.597 → 0.265")
print("  埋め込み      : 1.000 → 0.955")
print("\n文字の一致は層3で崩れるが、意味グループを共有する埋め込みだけは証拠を保つ。")
print("一方、埋め込みも余計な語を平均すると薄まる。単一特徴ではなく、複数の証拠を分類器へ渡す。")

<!-- PAIR_CLASSIFIER_SECTION_UNIT06 -->
## 実務の一連処理: ペア特徴 → 同一確率 → F1閾値 → group ID

`LogisticRegression` は数値特徴から2クラスの確率を学習する分類器です。C#でいえば、比較結果を並べたDTOを受け取り `PredictProbability` するサービスに相当します。

候補ペアごとに `(名前類似度, 価格差, cosine類似度)` を作り、`fit(X, y)` で学習します。`predict_proba(X)[:, 1]` は「同一entity」側の確率です。検証ラベルだけでF1が最大の閾値を決め、最後に閾値以上のペアをUnion-Findで推移的にまとめます。

In [ ]:
# STEP 1: 極小ペア表から3列の比較特徴を作る
from difflib import SequenceMatcher
from sklearn.linear_model import LogisticRegression

pair_names = ["apple pro", "apple-pro", "banana", "banana"]
pair_prices = np.array([100, 102, 300, 305], dtype=float)
pair_embeddings = np.array([[1, 0], [0.98, 0.02], [0, 1], [0.02, 0.98]], dtype=float)
candidate_pairs = [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]
pair_labels = np.array([1, 0, 0, 0, 0, 1])

pair_feature_rows = []
for left, right in candidate_pairs:
    name_score = SequenceMatcher(None, pair_names[left], pair_names[right]).ratio()
    price_gap = abs(pair_prices[left] - pair_prices[right])
    denominator = np.linalg.norm(pair_embeddings[left]) * np.linalg.norm(pair_embeddings[right])
    cosine_score = float(np.dot(pair_embeddings[left], pair_embeddings[right]) / denominator) if denominator else 0.0
    pair_feature_rows.append([name_score, price_gap, cosine_score])
pair_X = np.asarray(pair_feature_rows)
print("pair_X shape:", pair_X.shape)

# STEP 2: 同一ペア確率を学習し、検証F1で閾値を選ぶ
pair_model = LogisticRegression(random_state=42).fit(pair_X, pair_labels)
pair_probabilities = pair_model.predict_proba(pair_X)[:, 1]
best_pair_threshold, best_pair_f1 = None, -1.0
for threshold in np.linspace(0.1, 0.9, 9):
    prediction = pair_probabilities >= threshold
    tp = int(((prediction == 1) & (pair_labels == 1)).sum())
    fp = int(((prediction == 1) & (pair_labels == 0)).sum())
    fn = int(((prediction == 0) & (pair_labels == 1)).sum())
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    if f1 > best_pair_f1:
        best_pair_threshold, best_pair_f1 = float(threshold), f1
print("best threshold / F1:", best_pair_threshold, round(best_pair_f1, 3))

# STEP 3: 閾値以上の辺をUnionし、各レコードのgroup IDへ変換する
matched_pairs = [pair for pair, score in zip(candidate_pairs, pair_probabilities) if score >= best_pair_threshold]
parent = list(range(len(pair_names)))
def pair_find(item):
    while parent[item] != item:
        parent[item] = parent[parent[item]]
        item = parent[item]
    return item
for left, right in matched_pairs:
    left_root, right_root = pair_find(left), pair_find(right)
    if left_root != right_root:
        parent[right_root] = left_root
roots = [pair_find(item) for item in range(len(pair_names))]
root_to_group = {root: group for group, root in enumerate(dict.fromkeys(roots))}
pair_group_ids = np.array([root_to_group[root] for root in roots])
print("matched pairs:", matched_pairs)
print("group IDs:", pair_group_ids)

---
## 振り返り(自己評価 + TIL)

以下に1〜2文ずつ、自分の言葉で書いてみよう。

**1. 今日学んだこと:**

> (ここに書く)

**2. 難しかったこと・まだあやふやなこと:**

> (ここに書く)

**3. Feynman チェック:**

- blocking で候補を減らすと、なぜ recall が下がり得る?
- TF-IDF cosine が同義語の言い換えに弱い理由を「語彙表の列」で説明できる?
- `(batch, seq_len)` の mask に `[:, :, None]` が必要な理由を shape で説明できる?

> (ここに書く)

---
## まとめ

| 概念 | 一言でいうと |
|---|---|
| entity resolution | レコードのペアへ「同じ実体か」を当てる二値分類 |
| blocking | `Dictionary<key, List<row>>` のバケツ内だけ比較し、$O(n^2)$ を現実的な候補数へ落とす |
| 編集距離 | 挿入・削除・置換の最小回数。既習の2次元 DP が直結する |
| Jaccard | 文字 n-gram 集合の共通部分 / 和集合。局所的な重なりを測る |
| TF-IDF cosine | 重要語の重なり。L2 正規化済みなら内積が cosine |
| SVD / LSA | `(n, vocab)` の疎な語彙軸を `(n, k)` の密な潜在軸へ圧縮する |
| embedding | 共通文字がなくても、意味の近い語を近いベクトルへ置く |
| mask 付き mean pooling | `(batch, seq, dim)` から padding を除外し `(batch, dim)` へ集約する |
| 閾値と連結成分 | pair 確率を辺へ変え、Union-Find / BFS でグループ化する。誤った1辺の伝播に注意 |

### この先どこで使うか

- **unit07** — 今日の `attention_mask` と `(batch, seq_len, dim)` は Transformer の入力でそのまま再登場する。
- **unit10** — blocking / 低次元化 / 閾値は、推論コストと精度を一緒に設計する材料になる。
- **実務** — Web 由来の商品・企業・顧客データを統合するとき、安い文字列特徴から意味埋め込みへ段階的に重くする。

**次は演習 `ex01_blocking_and_pairs` へ進もう。lesson.ipynb を見ながらで OK。**

| 演習 | 内容 |
|---|---|
| `ex01_blocking_and_pairs` | 複数キーの blocking と候補ペア生成 |
| `ex02_string_similarity` | 編集距離 DP・Jaccard・型番一致 |
| `ex03_cosine_and_pooling` | cosine・SVD・mask 付き pooling |
| `ex04_capstone` | ペア特徴 → 閾値 → 連結成分まで一気通貫 |